# 10 — Human-in-the-Loop

**Model:** DeepSeek-V3 via Nebius AI Studio
**Pattern:** Checkpointed Graph with Approval Gates
**Difficulty:** ⭐⭐⭐☆☆

---

## The Problem

Fully autonomous agents are powerful — and risky. An agent that can send emails, modify databases, or call external APIs can cause real damage if it makes a wrong decision.

**Human-in-the-Loop (HITL)** adds approval gates at critical decision points. The agent pauses, presents its intended action to a human, and only proceeds if approved.

LangGraph's `interrupt` mechanism is designed exactly for this: it pauses graph execution at a specified node, persists the state (so the graph can resume later), and waits for external input.

## Architecture

```
START
  │
  ▼
[Plan Action]         ← Agent decides what to do
  │
  ▼
[⏸️ INTERRUPT]        ← Human reviews the plan
  │                     (graph paused, state saved)
  ├── Approved ──▶ [Execute Action] ──▶ END
  │
  └── Rejected  ──▶ [Revise Plan] ──▶ [⏸️ INTERRUPT again]
```

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict, Optional, List

llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)

checkpointer = MemorySaver()
print("Setup complete.")

## State

In [ ]:
class EmailState(TypedDict):
    task: str
    draft: str
    human_feedback: str
    revision_count: int
    approved: bool
    sent: bool
    action_log: List[str]

## Nodes

In [ ]:
def draft_email_node(state: EmailState) -> dict:
    is_revision = state["revision_count"] > 0
    context = (
        f"Previous draft:\n{state['draft']}\n\nHuman feedback: {state['human_feedback']}"
        if is_revision else ""
    )
    action = "Revising" if is_revision else "Drafting"
    print(f"\n[Agent] {action} email...")

    response = llm.invoke([
        SystemMessage(content=(
            "You are a professional email writer. "
            "Write clear, concise, professional emails. "
            "Always include Subject, To (placeholder), and Body."
        )),
        HumanMessage(content=(
            f"Task: {state['task']}\n\n{context}\n\n"
            f"{'Revise the email addressing the feedback above.' if is_revision else 'Write the email.'}"
        ))
    ])
    print(f"[Agent] Draft ready ({len(response.content)} chars).")
    return {
        "draft": response.content,
        "revision_count": state["revision_count"] + (1 if is_revision else 0),
        "action_log": state["action_log"] + [
            f"Draft {'revised' if is_revision else 'created'} (revision #{state['revision_count']})"
        ],
    }


def human_review_node(state: EmailState) -> dict:
    print(f"\n{'='*60}")
    print("HUMAN REVIEW REQUIRED")
    print(f"{'='*60}")
    print(state["draft"])
    print(f"{'='*60}")

    human_response = interrupt({
        "message": "Review the email draft above.",
        "options": {
            "approve": "Send the email as-is",
            "<feedback>": "Provide feedback for revision",
            "cancel": "Cancel the email"
        }
    })

    decision = human_response.strip().lower()
    if decision == "approve":
        print("\n[Human] Approved.")
        return {"approved": True, "human_feedback": ""}
    elif decision == "cancel":
        print("\n[Human] Cancelled.")
        return {"approved": False, "human_feedback": "cancelled"}
    else:
        print(f"\n[Human] Revision requested: {decision}")
        return {"approved": False, "human_feedback": human_response}


def send_email_node(state: EmailState) -> dict:
    print("\n[Agent] Sending email (simulated)...")
    return {"sent": True, "action_log": state["action_log"] + ["Email sent"]}


def route_after_review(state: EmailState) -> str:
    if state.get("human_feedback") == "cancelled":
        return "end"
    if state.get("approved"):
        return "send"
    return "revise" 

## Build the Graph

In [ ]:
builder = StateGraph(EmailState)
builder.add_node("draft", draft_email_node)
builder.add_node("review", human_review_node)
builder.add_node("send", send_email_node)

builder.set_entry_point("draft")
builder.add_edge("draft", "review")
builder.add_conditional_edges(
    "review",
    route_after_review,
    {"send": "send", "revise": "draft", "end": END}
)
builder.add_edge("send", END)

graph = builder.compile(checkpointer=checkpointer)
print("Human-in-the-Loop graph compiled.")

## Live Demo

In [ ]:
config = {"configurable": {"thread_id": "email-demo-01"}}

initial_state = {
    "task": (
        "Write a follow-up email to a potential client (Ahmed at TechCorp) "
        "after a product demo yesterday. We showed our AI analytics platform. "
        "Mention the 14-day free trial offer and offer to answer questions."
    ),
    "draft": "", "human_feedback": "", "revision_count": 0,
    "approved": False, "sent": False, "action_log": []
}

print("Starting agent — will pause for human review...")
result = graph.invoke(initial_state, config=config)
print("\nGraph paused — awaiting human input.")

In [ ]:
# Simulate human requesting a revision
result = graph.invoke(
    Command(resume="Make it shorter — max 3 sentences in the body. Add a specific call-to-action."),
    config=config
)
print("\nGraph paused again — awaiting approval of revised draft.")

In [ ]:
# Simulate human approving
result = graph.invoke(Command(resume="approve"), config=config)

print("\n" + "="*60)
print("ACTION LOG")
print("="*60)
for entry in result["action_log"]:
    print(f"  • {entry}")

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **`interrupt()`** | Pauses graph mid-node; state is serialized and saved |
| **`Command(resume=...)`** | Resumes a paused graph with human input |
| **Thread ID** | Same thread = same conversation; different thread = new session |
| **MemorySaver** | In-process only; swap for `SqliteSaver` in production |

## What's Next

**Notebook 11** builds a 3-tier hierarchical agent system.